# fuselab: fused LLM inference kernels in Triton (step 1 RMSNorm, step 2 add + RMSNorm + INT8)

RMSNorm runs dozens of times for every token an LLM generates. Plain PyTorch does it in
several passes over GPU memory; this kernel reads each row once, does all the maths on-chip,
and writes once. This notebook:

1. checks the kernel matches PyTorch (fp32 reference) across shapes and dtypes,
2. benchmarks it against eager PyTorch, `F.rms_norm`, `torch.compile` and Liger-Kernel,
3. plots effective memory bandwidth against the GPU's peak.

**To run on Kaggle:** Settings → Accelerator → **GPU T4 x2** (or P100), Internet **on**, then *Run All*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, triton
print("torch", torch.__version__, "| triton", triton.__version__, "| cuda", torch.version.cuda)

In [ ]:
!pip install -q --no-deps liger-kernel

In [ ]:
!mkdir -p fuselab tests bench results

In [ ]:
%%writefile fuselab/__init__.py
from .add_rmsnorm_quant import add_rmsnorm_quant, add_rmsnorm_quant_reference
from .rmsnorm import rmsnorm, rmsnorm_reference

__all__ = ["add_rmsnorm_quant", "add_rmsnorm_quant_reference", "rmsnorm", "rmsnorm_reference"]

In [ ]:
%%writefile fuselab/rmsnorm.py
"""Fused RMSNorm in Triton (inference, forward only).

y = x / sqrt(mean(x^2) + eps) * w, computed per row in fp32 and written back in x's dtype.

The point of fusing: the unfused PyTorch version makes several passes over x in GPU
memory (square, mean, rsqrt, multiply, multiply). This kernel reads each row once,
does all the maths in registers, and writes the result once.
"""

import torch
import triton
import triton.language as tl

MAX_COLS = 16384  # one row per program; wider rows would spill registers


@triton.jit
def _rmsnorm_kernel(
    x_ptr,
    w_ptr,
    out_ptr,
    x_row_stride,
    out_row_stride,
    n_cols,
    eps,
    BLOCK: tl.constexpr,
):
    row = tl.program_id(0)  # one program per row (token)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols

    # one trip in
    x = tl.load(x_ptr + row * x_row_stride + cols, mask=mask, other=0.0).to(tl.float32)
    w = tl.load(w_ptr + cols, mask=mask, other=0.0).to(tl.float32)

    # all the maths stays on-chip
    mean_sq = tl.sum(x * x, axis=0) / n_cols
    y = x * tl.rsqrt(mean_sq + eps) * w

    # one trip out (stored in out's dtype)
    tl.store(out_ptr + row * out_row_stride + cols, y, mask=mask)


def rmsnorm(x: torch.Tensor, w: torch.Tensor, eps: float = 1e-6, num_warps: int | None = None) -> torch.Tensor:
    """RMSNorm over the last dimension of ``x`` with weight ``w``."""
    if x.shape[-1] != w.shape[0]:
        raise ValueError(f"last dim of x ({x.shape[-1]}) != len(w) ({w.shape[0]})")
    n_cols = x.shape[-1]
    if n_cols > MAX_COLS:
        raise ValueError(f"hidden size {n_cols} > {MAX_COLS} is not supported by this kernel")

    x2 = x.reshape(-1, n_cols)
    if x2.stride(-1) != 1:
        x2 = x2.contiguous()
    w = w.contiguous()
    out = torch.empty_like(x2)

    block = triton.next_power_of_2(n_cols)
    if num_warps is None:
        num_warps = min(max(block // 256, 1), 8)

    _rmsnorm_kernel[(x2.shape[0],)](
        x2, w, out, x2.stride(0), out.stride(0), n_cols, eps, BLOCK=block, num_warps=num_warps
    )
    return out.reshape(x.shape)


def rmsnorm_reference(x: torch.Tensor, w: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Plain PyTorch, fp32 maths, same formula. Used for correctness and as the 'eager' baseline."""
    xf = x.float()
    y = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + eps) * w.float()
    return y.to(x.dtype)

In [ ]:
%%writefile fuselab/add_rmsnorm_quant.py
"""Fused residual add + RMSNorm + per-token INT8 quantisation (inference, forward only).

What a serving engine runs before every INT8 matmul, in one pass:

    residual = x + residual                      (written back in place, in the input dtype)
    y        = residual / sqrt(mean(residual^2) + eps) * w        (fp32)
    scale    = max(|y|) / 127                    (one fp32 per row / token)
    q        = round(y / scale)                  (int8, round half away from zero)

Unfused PyTorch makes about eight passes over the row in GPU memory (add, store, square, mean,
rsqrt, multiply, multiply, abs-max, divide, round, cast). This kernel reads x and residual once and
writes residual, q and scale once.
"""

from __future__ import annotations

import torch
import triton
import triton.language as tl

MAX_COLS = 16384
QMAX = 127.0
MIN_AMAX = 1e-8  # a row of zeros gets a tiny scale instead of dividing by zero


@triton.jit
def _add_rmsnorm_quant_kernel(
    x_ptr,
    res_ptr,
    w_ptr,
    q_ptr,
    scale_ptr,
    x_stride,
    res_stride,
    q_stride,
    n_cols,
    eps,
    BLOCK: tl.constexpr,
):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols

    # one trip in
    x = tl.load(x_ptr + row * x_stride + cols, mask=mask, other=0.0)
    res = tl.load(res_ptr + row * res_stride + cols, mask=mask, other=0.0)
    r = x + res  # in the input dtype, like the unfused add
    tl.store(res_ptr + row * res_stride + cols, r, mask=mask)  # residual updated in place

    rf = r.to(tl.float32)
    w = tl.load(w_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    y = rf * tl.rsqrt(tl.sum(rf * rf, axis=0) / n_cols + eps) * w

    amax = tl.maximum(tl.max(tl.abs(y), axis=0), 1e-8)
    v = y * (127.0 / amax)
    qf = tl.where(v >= 0, tl.floor(v + 0.5), tl.ceil(v - 0.5))  # round half away from zero
    qf = tl.minimum(tl.maximum(qf, -127.0), 127.0)

    # one trip out
    tl.store(q_ptr + row * q_stride + cols, qf.to(tl.int8), mask=mask)
    tl.store(scale_ptr + row, amax / 127.0)


def add_rmsnorm_quant(
    x: torch.Tensor, residual: torch.Tensor, w: torch.Tensor, eps: float = 1e-6, num_warps: int | None = None
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Returns (q int8 [..., N], scale fp32 [..., 1], residual). ``residual`` is updated in place."""
    if x.shape != residual.shape or x.dtype != residual.dtype:
        raise ValueError("x and residual must have the same shape and dtype")
    n = x.shape[-1]
    if w.shape[0] != n:
        raise ValueError(f"last dim of x ({n}) != len(w) ({w.shape[0]})")
    if n > MAX_COLS:
        raise ValueError(f"hidden size {n} > {MAX_COLS} is not supported by this kernel")
    if not residual.is_contiguous():
        raise ValueError("residual must be contiguous (it is written in place)")

    x2 = x.reshape(-1, n)
    if x2.stride(-1) != 1:
        x2 = x2.contiguous()
    r2 = residual.view(-1, n)
    w = w.contiguous()
    q = torch.empty(x2.shape, dtype=torch.int8, device=x.device)
    scale = torch.empty((x2.shape[0], 1), dtype=torch.float32, device=x.device)

    block = triton.next_power_of_2(n)
    if num_warps is None:
        num_warps = min(max(block // 256, 1), 8)
    _add_rmsnorm_quant_kernel[(x2.shape[0],)](
        x2, r2, w, q, scale, x2.stride(0), r2.stride(0), q.stride(0), n, eps, BLOCK=block, num_warps=num_warps
    )
    return q.reshape(x.shape), scale.reshape(*x.shape[:-1], 1), residual


def add_rmsnorm_quant_reference(
    x: torch.Tensor, residual: torch.Tensor, w: torch.Tensor, eps: float = 1e-6
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Plain PyTorch, same formula and rounding. Does NOT modify ``residual``; returns the new one."""
    r = x + residual
    rf = r.float()
    y = rf * torch.rsqrt(rf.pow(2).mean(-1, keepdim=True) + eps) * w.float()
    amax = y.abs().amax(-1, keepdim=True).clamp_min(MIN_AMAX)
    v = y * (QMAX / amax)
    q = torch.where(v >= 0, torch.floor(v + 0.5), torch.ceil(v - 0.5)).clamp(-QMAX, QMAX).to(torch.int8)
    return q, amax / QMAX, r

In [ ]:
%%writefile tests/test_rmsnorm.py
"""Correctness of the fused RMSNorm against the fp32 PyTorch reference.

Runs on a CUDA GPU, or on CPU with TRITON_INTERPRET=1 (slow, small shapes only).
"""

import os

import pytest
import torch

from fuselab import rmsnorm, rmsnorm_reference

INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and not INTERPRET:
    pytest.skip("needs a CUDA GPU or TRITON_INTERPRET=1", allow_module_level=True)

DTYPES = [torch.float32, torch.float16]
if DEVICE == "cuda" and torch.cuda.is_bf16_supported():
    DTYPES.append(torch.bfloat16)

TOL = {torch.float32: 1e-5, torch.float16: 2e-3, torch.bfloat16: 1.6e-2}

ROWS = [1, 7, 64] if INTERPRET else [1, 7, 128, 4096]
COLS = [128, 1000, 2560] if INTERPRET else [128, 1000, 2560, 4096, 8192]


@pytest.mark.parametrize("dtype", DTYPES)
@pytest.mark.parametrize("rows", ROWS)
@pytest.mark.parametrize("cols", COLS)
def test_matches_reference(dtype, rows, cols):
    torch.manual_seed(0)
    x = torch.randn(rows, cols, device=DEVICE, dtype=dtype)
    w = torch.randn(cols, device=DEVICE, dtype=dtype)
    got = rmsnorm(x, w)
    want = rmsnorm_reference(x, w)
    assert got.dtype == x.dtype and got.shape == x.shape
    torch.testing.assert_close(got, want, atol=TOL[dtype], rtol=TOL[dtype])


def test_3d_input_and_non_contiguous_rows():
    torch.manual_seed(0)
    big = torch.randn(4, 6, 512, device=DEVICE)
    x = big[:, ::2, :]  # rows not contiguous in memory
    w = torch.randn(512, device=DEVICE)
    torch.testing.assert_close(rmsnorm(x, w), rmsnorm_reference(x, w), atol=1e-5, rtol=1e-5)


def test_rejects_wrong_weight_length():
    x = torch.randn(2, 64, device=DEVICE)
    with pytest.raises(ValueError):
        rmsnorm(x, torch.ones(32, device=DEVICE))

In [ ]:
%%writefile tests/test_add_rmsnorm_quant.py
"""Correctness of fused residual add + RMSNorm + INT8 quantisation against the PyTorch reference.

Runs on a CUDA GPU, or on CPU with TRITON_INTERPRET=1 (slow, small shapes only).
"""

import os

import pytest
import torch

from fuselab.add_rmsnorm_quant import add_rmsnorm_quant, add_rmsnorm_quant_reference

INTERPRET = os.environ.get("TRITON_INTERPRET") == "1"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and not INTERPRET:
    pytest.skip("needs a CUDA GPU or TRITON_INTERPRET=1", allow_module_level=True)

DTYPES = [torch.float32, torch.float16]
if DEVICE == "cuda" and torch.cuda.is_bf16_supported():
    DTYPES.append(torch.bfloat16)

ROWS = [1, 7, 64] if INTERPRET else [1, 7, 128, 4096]
COLS = [128, 1000, 2560] if INTERPRET else [128, 1000, 2560, 4096, 8192]


def _check(x, res, w):
    want_q, want_scale, want_res = add_rmsnorm_quant_reference(x, res, w)
    res_in = res.clone()
    q, scale, out_res = add_rmsnorm_quant(x, res_in, w)
    assert out_res is res_in, "residual is updated in place and returned"
    # the residual add happens in the input dtype, so it matches exactly
    torch.testing.assert_close(res_in, want_res, atol=0, rtol=0)
    torch.testing.assert_close(scale, want_scale, atol=1e-6, rtol=1e-4)
    assert q.dtype == torch.int8 and q.shape == x.shape and int(q.abs().max()) <= 127
    # fp32 rounding order can move a value sitting exactly on .5 by one step; nothing more
    diff = (q.int() - want_q.int()).abs()
    assert int(diff.max()) <= 1 and float((diff > 0).float().mean()) < 1e-3
    # and the int8 values are a faithful encoding of the normalised row
    y = (want_res.float() * torch.rsqrt(want_res.float().pow(2).mean(-1, keepdim=True) + 1e-6)) * w.float()
    assert float(((q.float() * scale) - y).abs().max()) <= float(scale.max()) * 0.5 + 1e-5


@pytest.mark.parametrize("dtype", DTYPES)
@pytest.mark.parametrize("rows", ROWS)
@pytest.mark.parametrize("cols", COLS)
def test_matches_reference(dtype, rows, cols):
    torch.manual_seed(0)
    x = torch.randn(rows, cols, device=DEVICE, dtype=dtype)
    res = torch.randn(rows, cols, device=DEVICE, dtype=dtype)
    w = torch.randn(cols, device=DEVICE, dtype=dtype)
    _check(x, res, w)


def test_zero_row_does_not_divide_by_zero():
    x = torch.zeros(3, 256, device=DEVICE)
    res = torch.zeros(3, 256, device=DEVICE)
    q, scale, _ = add_rmsnorm_quant(x, res, torch.ones(256, device=DEVICE))
    assert int(q.abs().max()) == 0 and torch.isfinite(scale).all()


def test_3d_input():
    torch.manual_seed(1)
    x = torch.randn(2, 5, 512, device=DEVICE)
    res = torch.randn(2, 5, 512, device=DEVICE)
    w = torch.randn(512, device=DEVICE)
    q, scale, _ = add_rmsnorm_quant(x, res.clone(), w)
    want_q, want_scale, _ = add_rmsnorm_quant_reference(x, res, w)
    assert q.shape == (2, 5, 512) and scale.shape == (2, 5, 1)
    assert int((q.int() - want_q.int()).abs().max()) <= 1


def test_rejects_bad_inputs():
    x = torch.randn(2, 64, device=DEVICE)
    with pytest.raises(ValueError):
        add_rmsnorm_quant(x, torch.randn(2, 32, device=DEVICE), torch.ones(64, device=DEVICE))
    with pytest.raises(ValueError):
        add_rmsnorm_quant(x, torch.randn(2, 64, device=DEVICE), torch.ones(32, device=DEVICE))

In [ ]:
%%writefile bench/bench_rmsnorm.py
"""Benchmark the fused Triton RMSNorm against honest baselines.

Baselines:
  eager          plain PyTorch, fp32 maths (several separate GPU kernels)
  F.rms_norm     PyTorch's built-in op
  torch.compile  PyTorch's compiler, which writes its own fused Triton kernel
  liger          Liger-Kernel's hand-written Triton RMSNorm
  fuselab        this repo

Metric: effective memory bandwidth (GB/s) = bytes that must move / time. RMSNorm is
memory-bound, so the ceiling is the GPU's peak memory bandwidth.

Usage: python bench/bench_rmsnorm.py [--dtype float16|bfloat16|float32] [--out results]
"""

import argparse
import csv
import os
import sys

import torch
import triton
import triton.testing

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
from fuselab import rmsnorm, rmsnorm_reference  # noqa: E402

# Peak DRAM bandwidth in GB/s (vendor spec sheets)
PEAK_GBPS = {
    "T4": 320,
    "P100": 732,
    "L4": 300,
    "A10": 600,
    "A100-SXM4-40GB": 1555,
    "A100-SXM4-80GB": 2039,
    "A100 80GB PCIe": 1935,
    "H100 80GB HBM3": 3350,
    "H100 PCIe": 2000,
    "RTX 4090": 1008,
    "RTX 5090": 1792,
}

HIDDEN = [2560, 4096]  # Qwen3-4B, Llama-3-8B
ROWS = [1, 16, 64, 256, 1024, 4096, 16384]  # tokens in the batch


def peak_for(name: str) -> float | None:
    for key, v in PEAK_GBPS.items():
        if key in name:
            return v
    return None


def build_impls(eps: float):
    impls = {"eager": lambda x, w: rmsnorm_reference(x, w, eps)}
    if hasattr(torch.nn.functional, "rms_norm"):
        impls["F.rms_norm"] = lambda x, w: torch.nn.functional.rms_norm(x, (x.shape[-1],), w, eps)
    # One compiled function per hidden size (build_impls is called per size), and a high recompile
    # limit: every new batch shape recompiles, and past the default limit (8) torch.compile silently
    # falls back to eager, which made it look as slow as eager in the first run.
    for name in ("recompile_limit", "cache_size_limit"):
        if hasattr(torch._dynamo.config, name):
            setattr(torch._dynamo.config, name, 64)
    torch._dynamo.reset()
    compiled = torch.compile(rmsnorm_reference, dynamic=False)
    impls["torch.compile"] = lambda x, w: compiled(x, w, eps)
    try:
        from liger_kernel.ops.rms_norm import LigerRMSNormFunction

        # "gemma" casting = all maths in fp32 with offset 0, i.e. the same formula as ours
        impls["liger"] = lambda x, w: LigerRMSNormFunction.apply(x, w, eps, 0.0, "gemma")
    except Exception as e:  # liger missing or API changed
        print(f"skipping liger: {type(e).__name__}: {e}")
    impls["fuselab"] = lambda x, w: rmsnorm(x, w, eps)
    return impls


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dtype", default="float16", choices=["float16", "bfloat16", "float32"])
    ap.add_argument("--out", default="results")
    args = ap.parse_args()
    if not torch.cuda.is_available():
        sys.exit("needs a CUDA GPU")

    dtype = getattr(torch, args.dtype)
    gpu = torch.cuda.get_device_name()
    peak = peak_for(gpu)
    eps = 1e-6
    impls = build_impls(eps)  # rebuilt per hidden size below
    print(f"GPU: {gpu} (peak {peak or '?'} GB/s) | torch {torch.__version__} | triton {triton.__version__} | {args.dtype}")

    rows_out = []
    for n in HIDDEN:
        impls = build_impls(eps)
        for m in ROWS:
            x = torch.randn(m, n, device="cuda", dtype=dtype)
            w = torch.randn(n, device="cuda", dtype=dtype)
            want = rmsnorm_reference(x, w, eps)
            moved = (2 * x.numel() + w.numel()) * x.element_size()  # read x + w, write y
            for name, fn in impls.items():
                try:
                    got = fn(x, w)
                    torch.testing.assert_close(got.float(), want.float(), atol=2e-2, rtol=2e-2)
                    ms = triton.testing.do_bench(lambda: fn(x, w), warmup=25, rep=100, return_mode="median")
                except Exception as e:
                    print(f"  {name:13s} n={n:5d} m={m:6d}  FAILED: {type(e).__name__}: {str(e)[:120]}")
                    continue
                gbps = moved / (ms * 1e-3) / 1e9
                rows_out.append(
                    {
                        "gpu": gpu,
                        "dtype": args.dtype,
                        "hidden": n,
                        "rows": m,
                        "impl": name,
                        "us": round(ms * 1e3, 2),
                        "gbps": round(gbps, 1),
                        "pct_peak": round(100 * gbps / peak, 1) if peak else "",
                    }
                )
                print(f"  {name:13s} n={n:5d} m={m:6d}  {ms*1e3:8.2f} us  {gbps:7.1f} GB/s")

    os.makedirs(args.out, exist_ok=True)
    tag = f"{gpu.replace(' ', '_').replace('/', '_')}_{args.dtype}"
    csv_path = os.path.join(args.out, f"rmsnorm_{tag}.csv")
    with open(csv_path, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=list(rows_out[0]))
        wr.writeheader()
        wr.writerows(rows_out)
    print(f"saved {csv_path}")

    # summary at the largest batch: speed-up over eager and % of peak
    print("\nlargest batch (rows=%d):" % ROWS[-1])
    for n in HIDDEN:
        sub = {r["impl"]: r for r in rows_out if r["hidden"] == n and r["rows"] == ROWS[-1]}
        base = sub.get("eager")
        for name, r in sub.items():
            speed = f"{base['us'] / r['us']:.2f}x vs eager" if base else ""
            print(f"  hidden={n:5d} {name:13s} {r['gbps']:7.1f} GB/s  {r['pct_peak']}% of peak  {speed}")

    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, len(HIDDEN), figsize=(6 * len(HIDDEN), 4.5), sharey=True)
        for ax, n in zip(axes, HIDDEN):
            for name in impls:
                pts = [(r["rows"], r["gbps"]) for r in rows_out if r["hidden"] == n and r["impl"] == name]
                if pts:
                    ax.plot(*zip(*pts), marker="o", label=name, linewidth=2.5 if name == "fuselab" else 1.2)
            if peak:
                ax.axhline(peak, color="black", linestyle="--", linewidth=1)
                ax.text(ROWS[0], peak * 1.02, f"peak {peak} GB/s", fontsize=8)
            ax.set_xscale("log", base=2)
            ax.set_xlabel("tokens in batch (rows)")
            ax.set_title(f"RMSNorm, hidden={n}, {args.dtype}")
            ax.grid(alpha=0.3)
        axes[0].set_ylabel("effective bandwidth (GB/s)")
        axes[-1].legend()
        fig.suptitle(gpu)
        fig.tight_layout()
        png = os.path.join(args.out, f"rmsnorm_{tag}.png")
        fig.savefig(png, dpi=130)
        print(f"saved {png}")
    except ImportError:
        print("matplotlib not installed; skipped the plot")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile bench/bench_add_rmsnorm_quant.py
"""Benchmark fused residual add + RMSNorm + INT8 quantisation.

Baselines:
  eager          plain PyTorch (about eight separate GPU kernels)
  torch.compile  PyTorch's compiler, which fuses it into its own Triton kernel(s)
  norm-only      residual add in PyTorch -> fuselab's RMSNorm kernel -> quantise in PyTorch
                 (what you get by fusing just the norm)
  fuselab        everything in one kernel

Metric: microseconds per call, and effective bandwidth = bytes that must move / time.
Bytes: read x and residual, write residual, read w, write int8 q and one fp32 scale per row.

Usage: python bench/bench_add_rmsnorm_quant.py [--dtype float16|bfloat16|float32] [--out results]
"""

import argparse
import csv
import os
import sys

import torch
import triton
import triton.testing

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))
from fuselab import add_rmsnorm_quant, add_rmsnorm_quant_reference, rmsnorm  # noqa: E402

PEAK_GBPS = {"T4": 320, "P100": 732, "L4": 300, "A10": 600, "A100-SXM4-40GB": 1555, "A100-SXM4-80GB": 2039,
             "H100 80GB HBM3": 3350, "H100 PCIe": 2000, "RTX 4090": 1008, "RTX 5090": 1792}
HIDDEN = [2560, 4096]
ROWS = [1, 16, 64, 256, 1024, 4096, 16384]
IMPLS = ["eager", "torch.compile", "norm-only", "fuselab"]


def peak_for(name):
    return next((v for k, v in PEAK_GBPS.items() if k in name), None)


def norm_only(x, res, w, eps):
    r = x + res
    y = rmsnorm(r, w, eps).float()
    amax = y.abs().amax(-1, keepdim=True).clamp_min(1e-8)
    v = y * (127.0 / amax)
    q = torch.where(v >= 0, torch.floor(v + 0.5), torch.ceil(v - 0.5)).clamp(-127, 127).to(torch.int8)
    return q, amax / 127.0, r


def build_impls(eps):
    for name in ("recompile_limit", "cache_size_limit"):  # each new shape recompiles; don't fall back to eager
        if hasattr(torch._dynamo.config, name):
            setattr(torch._dynamo.config, name, 64)
    torch._dynamo.reset()
    compiled = torch.compile(add_rmsnorm_quant_reference, dynamic=False)
    return {
        "eager": lambda x, r, w: add_rmsnorm_quant_reference(x, r, w, eps),
        "torch.compile": lambda x, r, w: compiled(x, r, w, eps),
        "norm-only": lambda x, r, w: norm_only(x, r, w, eps),
        "fuselab": lambda x, r, w: add_rmsnorm_quant(x, r, w, eps),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dtype", default="float16", choices=["float16", "bfloat16", "float32"])
    ap.add_argument("--out", default="results")
    a = ap.parse_args()
    if not torch.cuda.is_available():
        sys.exit("needs a CUDA GPU")
    dtype, gpu, eps = getattr(torch, a.dtype), torch.cuda.get_device_name(), 1e-6
    peak = peak_for(gpu)
    print(f"GPU: {gpu} (peak {peak or '?'} GB/s) | torch {torch.__version__} | triton {triton.__version__} | {a.dtype}")

    out = []
    for n in HIDDEN:
        impls = build_impls(eps)
        for m in ROWS:
            x = torch.randn(m, n, device="cuda", dtype=dtype)
            res = torch.randn(m, n, device="cuda", dtype=dtype)
            w = torch.randn(n, device="cuda", dtype=dtype)
            want_q, _, _ = add_rmsnorm_quant_reference(x, res, w, eps)
            # timing input: x = 0 keeps the in-place residual from drifting over thousands of calls
            # (same memory traffic, no overflow)
            x0 = torch.zeros_like(x)
            e = x.element_size()
            moved = 3 * m * n * e + n * e + m * n + 4 * m
            for name, fn in impls.items():
                try:
                    q = fn(x, res.clone(), w)[0]
                    assert int((q.int() - want_q.int()).abs().max()) <= 1, "int8 output differs from reference"
                    ms = triton.testing.do_bench(lambda: fn(x0, res, w), warmup=25, rep=100, return_mode="median")
                except Exception as err:
                    print(f"  {name:13s} n={n:5d} m={m:6d}  FAILED: {type(err).__name__}: {str(err)[:120]}")
                    continue
                gbps = moved / (ms * 1e-3) / 1e9
                out.append({"gpu": gpu, "dtype": a.dtype, "hidden": n, "rows": m, "impl": name,
                            "us": round(ms * 1e3, 2), "gbps": round(gbps, 1),
                            "pct_peak": round(100 * gbps / peak, 1) if peak else ""})
                print(f"  {name:13s} n={n:5d} m={m:6d}  {ms*1e3:8.2f} us  {gbps:7.1f} GB/s")

    os.makedirs(a.out, exist_ok=True)
    tag = f"{gpu.replace(' ', '_').replace('/', '_')}_{a.dtype}"
    path = os.path.join(a.out, f"add_rmsnorm_quant_{tag}.csv")
    with open(path, "w", newline="") as f:
        wr = csv.DictWriter(f, fieldnames=list(out[0]))
        wr.writeheader()
        wr.writerows(out)
    print(f"saved {path}")
    for n in HIDDEN:
        for m in (1, ROWS[-1]):
            sub = {r["impl"]: r for r in out if r["hidden"] == n and r["rows"] == m}
            base = sub.get("eager")
            for name, r in sub.items():
                sp = f"{base['us'] / r['us']:.2f}x vs eager" if base else ""
                print(f"  hidden={n:5d} rows={m:6d} {name:13s} {r['us']:8.2f} us {r['gbps']:7.1f} GB/s "
                      f"{r['pct_peak']}% of peak  {sp}")

    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(1, len(HIDDEN), figsize=(6 * len(HIDDEN), 4.5), sharey=True)
        for ax, n in zip(axes, HIDDEN):
            for name in IMPLS:
                pts = [(r["rows"], r["gbps"]) for r in out if r["hidden"] == n and r["impl"] == name]
                if pts:
                    ax.plot(*zip(*pts), marker="o", label=name, linewidth=2.5 if name == "fuselab" else 1.2)
            if peak:
                ax.axhline(peak, color="black", linestyle="--", linewidth=1)
                ax.text(ROWS[0], peak * 1.02, f"peak {peak} GB/s", fontsize=8)
            ax.set_xscale("log", base=2)
            ax.set_xlabel("tokens in batch (rows)")
            ax.set_title(f"add + RMSNorm + INT8, hidden={n}, {a.dtype}")
            ax.grid(alpha=0.3)
        axes[0].set_ylabel("effective bandwidth (GB/s)")
        axes[-1].legend()
        fig.suptitle(gpu)
        fig.tight_layout()
        png = os.path.join(a.out, f"add_rmsnorm_quant_{tag}.png")
        fig.savefig(png, dpi=130)
        print(f"saved {png}")
    except ImportError:
        print("matplotlib not installed; skipped the plot")


if __name__ == "__main__":
    main()

## 1. Correctness

In [ ]:
!python -m pytest -q tests

## 2. Benchmark

In [ ]:
!python bench/bench_rmsnorm.py --dtype float16 --out results

In [ ]:
!python bench/bench_add_rmsnorm_quant.py --dtype float16 --out results

## 3. Results

In [ ]:
import glob
import pandas as pd
from IPython.display import Image, display

for p in sorted(glob.glob("results/*.png")):
    display(Image(p))
for p in sorted(glob.glob("results/*.csv")):
    print(p)
    display(pd.read_csv(p).pivot_table(index=["hidden", "rows"], columns="impl", values="us"))